# FIFA, Titanic, and IPL — DataFrames in Action (Task 18)

Task 18 pushes the DataFrame toolkit into three genuinely different real-world datasets, and the variety is the point. The FIFA World Cup 2022 results demand dictionary-building loops and ranking; the Titanic passenger lists drill missing-value strategy; and the IPL matches dataset requires the whole craft — renaming franchises, computing per-team win percentages, parsing string-encoded player lists, and building a full points table complete with season positions.

The FIFA section begins with `pd.read_csv(url)` against a Google-Sheets-published CSV — a convenient trick when a dataset is small enough to be public. Q-1 asks for each team's **on-target percentage** (shots on target ÷ total attempts × 100), rounded to two decimals, returned as a dictionary sorted by value. The solution loops over `df['Team'].unique()`, sums the `Total Attempts` and `On Target` columns per team, rounds the ratio, stores it in a dict, and finally sorts that dict with `sorted(..., key=lambda x: x[1], reverse=True)`. Q-2 counts how many times each team appeared with `value_counts()` and then ranks those counts using `DataFrame.rank(method='first')` — a parameter explored in detail because the default `average` collapses ties into fractional ranks. Q-3 runs the standard health check — `info()`, `describe()`, `isna().sum()` — fills missing numeric cells with column means, drops duplicate rows permanently, and removes four unneeded columns. Q-4 adds a `Rank` column from the `Team` field, casts it to `np.int16`, sets it as the index, and sorts by that index.

The **Titanic** section compares two cleaning philosophies head-to-head: on dataset 1, rows with a missing `Age` are dropped (`dropna(subset=['Age'], inplace=True)`); on dataset 2, the same gaps are *filled* with the value 20. Which is right depends entirely on how much data you can afford to lose and what the missingness means — the notebook makes that decision visible rather than silent.

Finally the **IPL** section is the most substantial. Q-6 builds a per-team table of matches played, win %, home win %, and away win % (remembering `Team1` is the home team and excluding no-result matches), after applying the now-familiar franchise renames. Q-7 finds the venue with the most no-result games, Q-8 parses the string-formatted `Team1Players`/`Team2Players` fields to count each player's final appearances, and Q-9/Q-10 construct a full points table (2 points per win, 1 each for no-result) and then extend it with a `SeasonPosition` column marking Winner, Runner, Third, and Fourth — using `df.at[row, col] = value` for targeted cell edits.

## 0. Setup

Import `numpy` and `pandas`, and set `display.max_columns = None` so wide frames show every column.

In [ ]:
import numpy as np
import pandas as pd

# to visualize all the column, run the below code
pd.options.display.max_columns = None
# to show all the records, use the below code
# pd.options.display.max_rows = None

# 1. FIFA World Cup 2022

A per-match results table where each row is one team's performance in one match. Four questions exercise looping, ranking, cleaning, and index surgery.

## Fifa Worldcup - 2022 dataset.

You can find the dataset from [here](https://www.kaggle.com/datasets/sayanroy729/fifa-worldcup-2022-results).

Also you can directly use an url to read the dataset by using `pd.read_csv()` method. Check the below code cell.

## 1.0 Loading the dataset

`pd.read_csv(url)` reads the CSV published at the Google Sheets URL, and `df.head()` previews it. Each row records one team's side of a match: `Team`, `Against`, `Group`, `Goal`, `Possession (%)`, and — crucially for Q-1 — hidden further right, the shot columns `Total Attempts` and `On Target`. Because the frame is wide, one team appears in multiple rows (one per match), so per-team statistics require aggregation *across rows*, not a simple column read. This 'one row per team per event' grain is standard in sports data and is exactly why the next question loops and filters rather than grouping naively — you are computing a ratio-of-sums, not a mean-of-ratios. Always establish the row grain before choosing an aggregation strategy.

In [ ]:
# To get the details about the dataset, please visit
# https://www.kaggle.com/datasets/sayanroy729/fifa-worldcup-2022-results

url = "https://docs.google.com/spreadsheets/d/e/2PACX-1vT3D_x_4DS6d51LKJ7ze1sxT5WpV5uiSVOFYHLwBiGru6vFyVv5h5-83AwFjxWYiWfCDjDAaarHAV-k/pub?gid=0&single=true&output=csv"
df = pd.read_csv(url)
df.head()

   Sl. No  Match No.     Team      Against Group  Goal  Possession (%)  \
0       1          1    Qatar      Ecuador     A     0              40   
1       2          1  Ecuador        Qatar     A     2              46   
2       3          2  England         Iran     B     6              69   
3       4          2     Iran      England     B     2              20   
4       5          3  Senegal  Netherlands     A     0              39   

   Inside Penalty Area  Outside Penalty Area  Assists  Total Attempts  \
0                    0                     0        0               5   
1                    2                     0        1               5   
2                    6                     0        6              13   
3                    2                     0        1               8   
4                    0                     0        0              15   

   On Target  Off Target  Target in Penalty  Target from Outside  \
0          0           5                  2     

# Task Q-1 — On-Target Percentage per Team

**Goal:** For every team, compute total shots on target as a percentage of total attempts, round to 2 decimals, return as a dict keyed by team, sorted by the percentage value.

### `Q-1:` Use the football dataset. Find out the total percentages that each team made on target. Display the result as a python dictionary where the keys are the team list and the values are the percentage values. Round off the percentage values up to 2 decimal places.

*Help:*
- First, find out how many total teams are participated in this worldcup. For that, you can use `unique()` method on the column "Team" or "Against".
- Loop through the teams list that you have found in previous section, and then filter the dataset according to that. After filtering the dataset, find out total attempts sum and total on target sum.
- After getting these values, find out the percentage by total on target divided by total attempts and multiply by 100. And store to a python dictionary where the key will be the team name and the values will be the percentages. 
- At the end,sort the dictionary by the values (not by the keys) and print the result.



**Sample Output:**
```bash
{'Costa Rica': 54.55,
 'Cameroon': 51.85,
 'Ecuador': 48.15,
 'Argentina': 46.99,
 'Brazil': 45.56,
 'England': 45.0,
 'Portugal': 40.32,
 'Ghana': 40.0,
 'Netherlands': 39.02,
 'Korea Republic': 36.73,
 'Australia': 36.0,
 'Mexico': 34.88,
 'Croatia': 34.78,
 'Germany': 34.33,
 'France': 32.97,
 'Spain': 32.69,
 'Belgium': 32.35,
 'Serbia': 32.26,
 'Iran': 31.43,
 'Uruguay': 31.25,
 'United States': 31.11,
 'Saudi Arabia': 31.03,
 'Senegal': 30.77,
 'Denmark': 30.56,
 'Switzerland': 30.56,
 'Japan': 30.23,
 'Wales': 29.17,
 'Qatar': 28.57,
 'Morocco': 28.3,
 'Tunisia': 26.67,
 'Poland': 25.0,
 'Canada': 17.65}
```

## 1.1 Approach — filter, sum, divide, sort

The sample output reveals the target: a dictionary like `{'Costa Rica': 54.55, 'Cameroon': 51.85, ...}` ordered from highest to lowest.

Step 1 — enumerate the teams. `df['Team'].unique()` returns each name once; because every team also appears in the `Against` column, either list is complete, though using `Team` alone is sufficient since every team batted at least once.

Step 2 — accumulate. The loop filters `df[df['Team'] == team]`, sums `Total Attempts` and `On Target`, and stores `round(total_on_target / total_attempts * 100, 2)` in a dict. This is a **ratio of sums** (percentage over the whole tournament), which is the correct reading when a team played several matches — averaging each match's percentage would weight a one-shot innings equally with a ten-shot one.

Step 3 — sort by value. `sorted(d.items(), key=lambda x: x[1], reverse=True)` orders the pairs by the percentage (index 1), descending, and `dict(...)` rebuilds a dictionary. Note the subtlety the sample output exposes: a dict is not *ordered* by keys, so the ordering here comes entirely from the sorted insertion. Watch for **division by zero** if any team somehow recorded no attempts — guard with a conditional or `.replace(0, np.nan)`.

In [ ]:
# code here
all_teams = df['Team'].unique()
on_target_percentage = {}
for team in all_teams:
    total_attempts = df[df['Team'] == team]["Total Attempts"].sum()
    total_on_target = df[df['Team'] == team]["On Target"].sum()
    on_target_percentage[team] = round(total_on_target / total_attempts * 100, 2)
dict(sorted(on_target_percentage.items(), key = lambda x: x[1], reverse=True))

{'Costa Rica': 54.55,
 'Cameroon': 51.85,
 'Ecuador': 48.15,
 'Argentina': 46.15,
 'Brazil': 45.56,
 'England': 45.0,
 'Portugal': 40.32,
 'Ghana': 40.0,
 'Netherlands': 39.02,
 'Korea Republic': 36.73,
 'Australia': 36.0,
 'Mexico': 34.88,
 'France': 34.65,
 'Germany': 34.33,
 'Croatia': 33.73,
 'Spain': 32.69,
 'Belgium': 32.35,
 'Serbia': 32.26,
 'Iran': 31.43,
 'Uruguay': 31.25,
 'United States': 31.11,
 'Saudi Arabia': 31.03,
 'Senegal': 30.77,
 'Denmark': 30.56,
 'Switzerland': 30.56,
 'Japan': 30.23,
 'Wales': 29.17,
 'Qatar': 28.57,
 'Morocco': 28.33,
 'Tunisia': 26.67,
 'Poland': 25.0,
 'Canada': 17.65}

# Task Q-2 — Matches Played and Team Ranks

**Goal:** Count how many times each team played, then rank those counts. The note highlights `rank(method='first')` because the default `average` produces fractional ranks on ties.

### `Q-2:` Find out how many times the teams are played in this Fifa Worldcup-2022. On top of this, find out the ranks of the teams. 

Note: The `DataFrame.rank()` method takes an optiinal parameter named `method`. This parameter takes different values, but one of them is `average` which is by-default. So, when you do the rank, you will get some 2.5 like floating values. But if you change the value as `first`, then you will get in integers but the datatype will be float. So, try with `method="first"` parameter.

## 1.2 Approach — value_counts then rank

`df['Team'].value_counts()` counts appearances per team and returns a Series sorted by count descending. Chaining `.rank(method='first')` then assigns ranks to those counts — but here lies a classic trap.

The default `method='average'` gives **tied** entries the average of the ranks they span: if Morocco and Croatia both have 7 matches occupying ranks 29 and 30, average gives both 29.5. That is mathematically sensible but visually odd when you expect integers. `method='first'` breaks ties by order of appearance in the Series, assigning 29 to the first and 30 to the second; the dtype stays `float` even though the values look like integers. Other methods worth knowing: `'min'` gives tied entries the lowest rank (both 29), `'max'` the highest (both 30), and `'dense'` produces consecutive ranks with no gaps (both 29, next tie 30). Selecting the right method changes the resulting numbers for tied groups, which is precisely why the question calls out `'first'`. At the top of the output, teams that played deeper (finalists played 7) sit below the rank-1 entry, so check which end you are reading.

In [ ]:
# code here
df['Team'].value_counts().rank(method="first")

Morocco           29.0
Croatia           30.0
Argentina         31.0
France            32.0
England           25.0
Brazil            26.0
Netherlands       27.0
Portugal          28.0
Poland            17.0
Japan             18.0
Switzerland       19.0
Australia         20.0
Korea Republic    21.0
United States     22.0
Senegal           23.0
Spain             24.0
Cameroon           1.0
Uruguay            2.0
Belgium            3.0
Ghana              4.0
Canada             5.0
Qatar              6.0
Costa Rica         7.0
Germany            8.0
Ecuador            9.0
Mexico            10.0
Tunisia           11.0
Denmark           12.0
Saudi Arabia      13.0
Wales             14.0
Iran              15.0
Serbia            16.0
Name: Team, dtype: float64

# Task Q-3 — Inspect, Fill, Drop

**Goal:** Print dataset information and description; check and fill missing values with column means; drop duplicate rows permanently; remove the columns `Sl No`, `Match No.`, `Red Cards`, and `Pts` permanently.

### `Q-3:` Find out these below topics:
- The information about the Fifa worldcup dataset.
- The description about the Fifa worldcup dataset
- Check is there any missing values, if there is any missing values, fill that value with the average value for that particular column.
- Drop all the duplicate rows permanently.
- Drop the columns: "Sl No", "Match No.", "Red Cards" and "Pts" permanently. 

## 1.3 Approach — the cleaning checklist

`print(df.info())` shows the shape (128 rows, 38 columns), each column's dtype, and non-null counts — the number to hide in is the first column over the frame. `print(df.describe())` gives numeric summary statistics (mean, std, quartiles) for the numeric columns, useful for spotting impossible values. `print(df.isna().sum())` then lists the missing count per column, pinpointing what needs filling.

Filling with the column mean is `df.fillna(df.mean(numeric_only=True), inplace=True)` — means only make sense for numeric columns, hence the `numeric_only` guard (older Pandas used deprecation warnings for mixed dtypes). This is a blunt instrument: it distorts variance and assumes missingness is random, so `median` is often safer on skewed columns and dropping rows is better when data is abundant.

`df.drop_duplicates(inplace=True)` removes exact duplicate rows permanently, and `df.drop(columns=['Sl. No', 'Match No.', 'Red Cards', 'Pts'], inplace=True)` removes unneeded columns. Two care points: the sample shows `Sl No` while the actual header is `Sl. No` (with a period) — column names must match **exactly**; and `inplace=True` mutates the original frame, so if you still need the raw data, operate on `df.copy()` first. Dropping the sequence/index columns here is purely tidiness — they carry no analytical information.

In [ ]:
# code here
print(df.info())
print(df.describe())
print(df.isna().sum())
df.drop_duplicates(inplace=True)
df.drop(columns=["Sl. No", "Match No.", "Red Cards", "Pts"], inplace=True)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 128 entries, 0 to 127
Data columns (total 38 columns):
 #   Column                           Non-Null Count  Dtype 
---  ------                           --------------  ----- 
 0   Sl. No                           128 non-null    int64 
 1   Match No.                        128 non-null    int64 
 2   Team                             128 non-null    object
 3   Against                          128 non-null    object
 4   Group                            128 non-null    object
 5   Goal                             128 non-null    int64 
 6   Possession (%)                   128 non-null    int64 
 7   Inside Penalty Area              128 non-null    int64 
 8   Outside Penalty Area             128 non-null    int64 
 9   Assists                          128 non-null    int64 
 10  Total Attempts                   128 non-null    int64 
 11  On Target                        128 non-null    int64 
 12  Off Target                       128

# Task Q-4 — Add, Cast, Index, Sort the Rank

**Goal:** Add a `Rank` column holding the team rank, cast it to `np.int16`, make it the permanent index, and sort the frame by that index.

### `Q-4:` Do these below operations:
- Find out the rank based on the "Team" column and save the result by adding a new column named "Rank".
- Change the datatype of this column to integer by using `np.int16`
- Set the index of the DataFrame by using this "Rank" column permanently.
- After that, sort the dataframe based on the "Rank" index.

## 1.4 Approach — a typed, sorted rank index

First `df['Rank'] = df['Team'].rank(method='first')` creates the rank column aligned to each row's team appearance, and `df['Rank'] = df['Rank'].astype(np.int16)` narrows the dtype from `float64` to a compact 16-bit integer — sensible for rank values that are small, and a nice demonstration of memory-conscious casting.

`df.set_index('Rank', inplace=True)` promotes the rank to the row index permanently, and `df.sort_index()` returns the frame ordered by that index (ascending by default). The resulting view is grouped by rank: rank 1 holds Argentina's first appearance, rank 2 its second, and so on — though because the same team appears in many matches, its rows receive **consecutive ranks** rather than one shared rank, which is the direct consequence of `method='first'` in Q-2/Q-4. If you had wanted one row per team you would first aggregate; here the rank is per-appearance, so interpret accordingly. `sort_index()` is ascending by default, and `ascending=False` would reverse it; note that sorting by a string index would order alphabetically.

In [ ]:
# code here
df["Rank"] = df["Team"].rank(method='first')
df["Rank"] = df["Rank"].astype(np.int16)
df.set_index("Rank", inplace=True)
df.sort_index()

           Team        Against Group  Goal  Possession (%)  \
Rank                                                         
1     Argentina   Saudi Arabia     C     1              51   
2     Argentina         Mexico     C     2              50   
3     Argentina         Poland     C     2              67   
4     Argentina      Australia     C     2              53   
5     Argentina    Netherlands     C     2              44   
...         ...            ...   ...   ...             ...   
124     Uruguay       Portugal     H     0              35   
125     Uruguay          Ghana     H     2              39   
126       Wales  United States     B     1              39   
127       Wales           Iran     B     0              51   
128       Wales        England     B     0              33   

      Inside Penalty Area  Outside Penalty Area  Assists  Total Attempts  \
Rank                                                                       
1                       1                

# 2. Titanic — Missing Value Strategy

Two nearly identical Titanic datasets are used to contrast a cleaning decision: drop the rows, or fill the gaps?

## Questions on Titanic dataset.

You can get the dataset from [here](https://www.kaggle.com/competitions/titanic). This is the competition page on Kaggle. To download the dataset from here, I guess you have to register for the compition. So, do so and then download the dataset.

Also, for now you can use this url to read the dataset like before: 
- dataset 1: https://docs.google.com/spreadsheets/d/e/2PACX-1vQjh5HzZ1N0SU7ME9ZQRzeVTaXaGsV97rU8R7eAcg53k27GTstJp9cRUOfr55go1GRRvTz1NwvyOnuh/pub?gid=1562145139&single=true&output=csv
- dataset 2: https://docs.google.com/spreadsheets/d/e/2PACX-1vQcPvQsSC9aNFogvbUG08nu0bGHlOclGYaOlhND_LE5Ff7ZnHQ5VYzAgpyT5XNklgiT54SsNgHePsUa/pub?gid=1656109608&single=true&output=csv

### `Q-5:` Do the below tasks:
1. With dataset 1, drop those records which only have missing values of the "Age" column permanently.

2. With the dataset 2, fill the missing values with 20 to the only "Age" column permanently.

## 2.1 Drop vs fill for missing ages (Q-5)

Both datasets are read from published Google Sheets CSVs. The question then does two opposite things with the same `Age` column, deliberately.

**Dataset 1 — drop.** `titanic_df1.dropna(subset=['Age'], inplace=True)` removes every row whose `Age` is missing, leaving only passengers with a known age. The `subset=['Age']` is essential: without it, `dropna` would delete any row missing a value in **any** column, often wiping out far more data than intended. Dropping is honest — you analyse only known ages — but it sacrifices sample size and can bias the result if younger or older passengers were less likely to have their age recorded.

**Dataset 2 — fill.** `titanic_df2['Age'] = titanic_df2['Age'].fillna(20)` keeps every row but substitutes the constant 20 for missing ages. That preserves the row count but injects a strong, arbitrary assumption (everyone unknown is exactly 20). A mean or median fill would be more defensible, and a model-based fill more sophisticated still.

The lesson is not 'drop is right' or 'fill is right' — it is that missing-data handling is a **decision with consequences** you must choose consciously and document. The same codebase might drop when data is plentiful and fill when it is scarce.

In [ ]:
# code here
titanic_df1 = pd.read_csv("https://docs.google.com/spreadsheets/d/e/2PACX-1vQjh5HzZ1N0SU7ME9ZQRzeVTaXaGsV97rU8R7eAcg53k27GTstJp9cRUOfr55go1GRRvTz1NwvyOnuh/pub?gid=1562145139&single=true&output=csv")
titanic_df2 = pd.read_csv("https://docs.google.com/spreadsheets/d/e/2PACX-1vQcPvQsSC9aNFogvbUG08nu0bGHlOclGYaOlhND_LE5Ff7ZnHQ5VYzAgpyT5XNklgiT54SsNgHePsUa/pub?gid=1656109608&single=true&output=csv")

titanic_df1.dropna(subset=["Age"], inplace=True)
titanic_df2["Age"] = titanic_df2["Age"].fillna(20)

# 3. IPL Matches

One row per match, with columns including `Team1` (home), `Team2` (away), `WinningTeam`, `WonBy`, `Venue`, `Season`, `MatchNumber`, and the string-encoded `Team1Players`/`Team2Players`. This section builds a full analytics stack on top of it.

## Questions on the IPL dataset

matches dataset = https://drive.google.com/file/d/1yKVUuexl6lIKuFQy7uIPgDgXhJ0L4SIg/view?usp=share_link

Code to directly use in colab
```
ipl_matches = "https://docs.google.com/spreadsheets/d/e/2PACX-1vRy2DUdUbaKx_Co9F0FSnIlyS-8kp4aKv_I0-qzNeghiZHAI_hw94gKG22XTxNJHMFnFVKsO4xWOdIs/pub?gid=1655759976&single=true&output=csv"

```




### `Q-6:` Make a dataframe of each team in IPL with details like - Team Name, Matches Played, Win%, Home Win%, Away Win%.
Show sorted dataframe on Win%

Replace old team name as new name before performing any tasks.
```
Delhi Daredevils ->Delhi Capitals
Kings XI Punjab -> Punjab Kings
Rising Pune Supergiants -> Rising Pune Supergiant
```

Note: Team1 represents Home team. Exclude not result matches.


## 3.1 Load and rename (Q-6, part 1)

`pd.read_csv(ipl_matches)` reads the season-long match log, and then the franchise renames are applied exactly as in Task 17: a dict maps `Delhi Daredevils → Delhi Capitals`, `Kings XI Punjab → Punjab Kings`, and `Rising Pune Supergiants → Rising Pune Supergiant`, and `matches.replace(dict.keys(), dict.values(), inplace=True)` applies them frame-wide across `Team1`, `Team2`, and `WinningTeam`.

Doing this *first*, before any counting, is the whole point: if you computed win percentages before renaming, Delhi Capitals' history would be split across two names and both records would be wrong. Data cleaning precedes analysis — always. Also note `WonBy` versus `WinningTeam`: the former records the *method* (`Runs`, `Wickets`, `NoResults`), the latter the winner; the distinction matters for both the win calculations and the no-result questions.

In [ ]:
# code here
ipl_matches = "https://docs.google.com/spreadsheets/d/e/2PACX-1vRy2DUdUbaKx_Co9F0FSnIlyS-8kp4aKv_I0-qzNeghiZHAI_hw94gKG22XTxNJHMFnFVKsO4xWOdIs/pub?gid=1655759976&single=true&output=csv"
matches = pd.read_csv(ipl_matches)
changed_name = {'Delhi Daredevils':'Delhi Capitals', 
                'Kings XI Punjab':'Punjab Kings',
                'Rising Pune Supergiants':'Rising Pune Supergiant'}
matches.replace(changed_name.keys(), changed_name.values(),inplace=True )

## 3.2 Per-team win record (Q-6, part 2)

The task requires, per team: matches played, win %, home win %, and away win %. The solution builds these in a loop and assembles a fresh frame.

- `played = df[(df.Team1 == team) | (df.Team2 == team)].shape[0]` — matches in which the team appeared in either slot.
- `wins = df[df.WinningTeam == team].shape[0]` — matches it won.
- `home_win` uses only matches where the team was `Team1` (home) in the denominator, and counts when it won.
- `away_win` does the same for `Team2` (away).

Two details the question flags. First, `Team1` represents the **home** team, so home and away percentages are correctly separated by slot. Second, **no-result matches must be excluded** — `df = matches[~matches.WinningTeam.isna()]` first drops rows where there was no winner, otherwise they would deflate every win percentage. The loop appends `[team, played, wins/played*100, home_win, away_win]` per team into a list, and the list is assigned to a set of new column names, after which `new_df.sort_values('Win Percentage', ascending=False, inplace=True)` orders the table. The output shows Gujarat Titans (75%) and Lucknow Super Giants (60%) at the top of the short-history franchises, with Chennai Super Kings (58.2%) and Mumbai Indians (56.7%) leading the long-history sides. The loop is clear but slow; a fully vectorised groupby formulation would scale better on millions of rows.

In [ ]:
df = matches[~matches.WinningTeam.isna()]
new_df = pd.DataFrame()
teams = df.Team1.unique()
data = []
for team in teams:
    played = df[(df.Team1 == team) | (df.Team2 == team)].shape[0]
    wins = df[df.WinningTeam == team].shape[0]
    home_win = df[(df.WinningTeam == team) | (df.Team1 == team)].shape[0] / df[df.Team1 == team].shape[0] * 100
    away_win = df[(df.WinningTeam == team) | (df.Team2 == team)].shape[0] / df[df.Team2 == team].shape[0] * 100
    data.append([team, played, wins/played*100, home_win, away_win])
new_df[["Team", "Matches Played", "Win Percentage", "Home Win Percentage", "Away Win Percentage"]] = data
new_df.sort_values("Win Percentage", inplace=True, ascending=False)
new_df

                           Team Matches Played Win Percentage  \
5                Gujarat Titans             16           75.0   
6          Lucknow Super Giants             15           60.0   
4           Chennai Super Kings            208      58.173077   
9                Mumbai Indians            231      56.709957   
7         Kolkata Knight Riders            223      51.121076   
0              Rajasthan Royals            190      50.526316   
10       Rising Pune Supergiant             30           50.0   
2           Sunrisers Hyderabad            152      49.342105   
1   Royal Challengers Bangalore            223      48.878924   
3                Delhi Capitals            222      46.396396   
8                  Punjab Kings            218      46.330275   
11                Gujarat Lions             30      43.333333   
14         Kochi Tuskers Kerala             14      42.857143   
13              Deccan Chargers             75      38.666667   
12                Pune Wa

# Task Q-7 — Venue with Most No-Result Matches

**Goal:** Identify the venue that hosted the most 'no result' matches.

### `Q-7:` Venues with most "no result" matches.

## 3.3 Approach — filter, count, take the top

No-result matches are flagged in the `WonBy` column with the value `'NoResults'`. The one-liner `matches[matches.WonBy == 'NoResults'].Venue.value_counts().index[0]` filters to those rows, counts occurrences per `Venue`, and reads the first index of the descending `value_counts()` Series — the most frequent venue, which comes out as **'M Chinnaswamy Stadium'**.

This is the 'filter → value_counts → first index' idiom for a top-1 categorical answer, and it generalises: `.head(7)` (as in Task 17's cities question) gives the top seven, and `.idxmax()` is the equivalent direct call on the value counts. A couple of caveats worth internalising: `value_counts()` excludes `NaN` by default (pass `dropna=False` to include missing venues), and ties are broken by the Series order, so if two venues shared the maximum the result would be somewhat arbitrary. For a rigorous answer you might print `.head()` rather than only `[0]` and note any near-ties. Rain-affected venues with poor drainage naturally accumulate no-results, which makes this a real, interpretable insight rather than a curiosity.

In [ ]:
# code here
matches[matches.WonBy == "NoResults"].Venue.value_counts().index[0]

'M Chinnaswamy Stadium'

# Task Q-8 — Most Appearances in Finals

**Goal:** Across all final matches, find the player who appeared most often. The `Team1Players` and `Team2Players` columns hold the squads as **strings**, not lists, so they must be parsed.

### `Q-8:` Player with most appearance in final match.

`Team1Players` and `Team2Players` have all players name. It is not a list of players name instead it is str. So handle it as string.

Hint: split and strip will help; Make a series of all players in final and do value counts


## 3.4 Approach — parse the pseudo-list strings

The squad fields look like `"['Player A', 'Player B', ...]"` — a Python-list *repr* stored as text. The helper `getPlayers(l)` strips the outer brackets and quotes and splits on `", "`:
```python
def getPlayers(l):
    return pd.Series(list(map(lambda x: x.strip("'"), l.lstrip("'[").rstrip("]'").split(', '))))
```
`lstrip("'[")` and `rstrip("]'")` remove the framing characters, `.split(', ')` separates the names, and the inner `strip("'")` cleans any residual quote from each. The result is a Series of player names.

The code filters to finals (`matches[matches['MatchNumber'] == 'Final']`), then loops over every final's `Team1Players` and `Team2Players`, appending each parsed Series to one growing `playerSeries`, and finally `.value_counts()` ranks the players. The point of the exercise is handling **structured text** without a real parser — a common situation in scraped data. Two modern cautions: the notebook warns `FutureWarning` that `pd.Series.append` is deprecated in favour of `pd.concat`, and constructing an empty `pd.Series()` now triggers a dtype warning; both would be written with `pd.concat(..., ignore_index=True)` today. The parsed result lists the players who featured in the most finals — a nice proxy for sustained excellence.

In [ ]:
# code here

def getPlayers(l):
    return pd.Series(list(map( lambda x: x.strip("'"), l.lstrip("'[").rstrip("]'").split(", "))))

df_final = matches[matches["MatchNumber"] == "Final"]
playerSeries = pd.Series()
for player_list in df_final.Team1Players:
    x = getPlayers(player_list)
    playerSeries = playerSeries.append(x)
for player_list in df_final.Team2Players:
    x = getPlayers(player_list)
    playerSeries = playerSeries.append(x)
playerSeries.value_counts().head()

C:\Users\rajti\AppData\Local\Temp\ipykernel_10576\3753546966.py:7: FutureWarning: The default dtype for empty Series will be 'object' instead of 'float64' in a future version. Specify a dtype explicitly to silence this warning.
  playerSeries = pd.Series()
C:\Users\rajti\AppData\Local\Temp\ipykernel_10576\3753546966.py:10: FutureWarning: The series.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  playerSeries = playerSeries.append(x)
C:\Users\rajti\AppData\Local\Temp\ipykernel_10576\3753546966.py:10: FutureWarning: The series.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  playerSeries = playerSeries.append(x)
C:\Users\rajti\AppData\Local\Temp\ipykernel_10576\3753546966.py:10: FutureWarning: The series.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  playerSeries = playerSeries.append(x)
C:\Users\rajti\Ap

MS Dhoni     10
SK Raina      8
AT Rayudu     7
R Ashwin      7
DJ Bravo      7
dtype: int64

# Task Q-9 — IPL Points Table

**Goal:** Write `point_table(season)` returning each team's matches played, matches won, no-results, and points — 2 per win, 1 to each side for a no-result — sorted by points descending, with `TeamName` as the index.

### `Q-9:` IPL Point Table

Make a function `point_table` which take `season` as parameter and show points table in non-ascendng order of points and in ascending order of team name.

For winning - 2 Ponits; 
For loosing - 0 Point
For not result both team gets 1 points.

Make dataframe which will have
`TeamName` `MatchesPlayed` `MatchesWon` `NoResult` `Points`
make `TeamName` as index.

```
season parametr should be one of these->
['2022', '2021', '2020/21', '2019', '2018', '2017', '2016', '2015', '2014', '2013', '2012', '2011', '2009/10', '2009', '2007/08']
```


Output of two Top 2 in season 2022
```
TeamName    MatchesPlayed	MatchesWon	NoResult	Points
				
Gujarat Titans	    16	   12	       0	     24
Rajasthan Royals	  17	   10	       0	     20

```

## 3.5 Approach — pure functions plus apply

The elegant structure here is three tiny helper functions — `matches_played(df, team)`, `matches_won(df, team)`, and `matches_no_result(df, team)` — each returning a scalar count. Writing them separately keeps the main function readable.

`point_table(season)` then:
1. filters the matches to the requested season (`df = matches[matches.Season == season]`),
2. creates a frame whose `Team Name` column is `df.Team1.unique()` (every team appears at least once as the home side, so this captures the full roster),
3. uses `.apply(lambda x: ...)` on that column to fill the played/won/no-result columns via the helpers,
4. computes `Points = Matches Won * 2 + No Result`,
5. `sort_values('Points', ascending=False)` and `set_index('Team Name')`.

For season 2022 the output leads with Gujarat Titans (16 played, 12 won, 0 no-results, 24 points) and Rajasthan Royals (17, 10, 0, 20). The `apply`-with-lambda pattern is expressive though row-wise loops are slower than vectorised alternatives; a `groupby` formulation would be preferable at scale. The design choice to `return` a DataFrame (rather than print) is precisely what lets Q-10 extend it.

In [ ]:
# code here
def matches_played(df, team):
    return df[(df.Team1 == team) | (df.Team2 == team)].shape[0]
def matches_won(df, team):
    return df[df.WinningTeam == team].shape[0]
def matches_no_result(df, team):
    return df[((df.Team1 == team) | (df.Team2 == team)) & (df.WinningTeam.isnull())].shape[0]

def point_table(season):
    df = matches[matches.Season == season]
    new_df = pd.DataFrame()
    new_df['Team Name'] = df.Team1.unique()
    new_df["Matches Played"] = new_df["Team Name"].apply(lambda x: matches_played(df, x))
    new_df["Matches Won"] = new_df["Team Name"].apply(lambda x: matches_won(df, x))
    new_df["No Result"] = new_df["Team Name"].apply(lambda x: matches_no_result(df, x))
    new_df["Points"] = new_df["Matches Won"]*2 + new_df["No Result"]
    new_df.sort_values("Points", ascending = False, inplace=True)
    new_df.set_index("Team Name", inplace=True)
    return new_df
point_table("2022")

                             Matches Played  Matches Won  No Result  Points
Team Name                                                                  
Gujarat Titans                           16           12          0      24
Rajasthan Royals                         17           10          0      20
Royal Challengers Bangalore              16            9          0      18
Lucknow Super Giants                     15            9          0      18
Delhi Capitals                           14            7          0      14
Punjab Kings                             14            7          0      14
Sunrisers Hyderabad                      14            6          0      12
Kolkata Knight Riders                    14            6          0      12
Chennai Super Kings                      14            4          0       8
Mumbai Indians                           14            4          0       8

# Task Q-10 — Season Position Extension

**Goal:** Extend the points table with a `SeasonPosition` column: Winner, Runner, 3 (losing Qualifier 2), and 4 (losing Eliminator); teams below the top four simply take their points rank. Use `df.at[row_index, col_label] = value` for individual cell edits.

### `Q-10:` IPL Point Table cont.
Extend the above IPL Point Table with an extra column as `SeasonPosition`

Team below top 4 after sorting on `Points` and then on `TeamName` Will have same `SeasonPosition` as there rank. use rank function.

Teams in Top four will have `SeasonPosition` as:
```
    'Winner' - Team won final
    'Runner' - Team lost Final
    3 - Losing Team in Qualifier2
    4 - Losing Team in Eliminator
```

For changing value of pariticular cell use `df.at[row_index, col_label] = value`

Output of two Top 2 in season 2022. Your result should have all teams.
```
TeamName    MatchesPlayed	MatchesWon	NoResult	Points   SeasonPosition
				
Gujarat Titans	    16	   12	       0	     24         Winner
Rajasthan Royals	  17	   10	       0	     20         Runner

```

Note: If you try to chnage value of view of any dataframe a warnig will be shown. To avoid it, make a copy of the dataframe you want to change in by `df.copy()`

## 3.6 Approach — rank by points, then overwrite the top four

`point_table_extension(season)` starts from `point_table(season)`, then seeds the position column with numeric ranks:
```python
df3['SeasonPosition'] = df3.Points.rank(ascending=False, method='first').astype('object')
```
`astype('object')` is essential — the column must hold both integers and text strings, and an integer column could never accept 'Winner'.

Next it identifies the four knockout losers. A `LoosingTeam` column is derived by appending, for each match, whichever team lost: `df[df.WinningTeam == df.Team1]['Team2']` gives Team2 when Team1 won, and vice versa. From the season's matches it extracts the **Final** (`wining_team = final.WinningTeam.values[0]`, `runner = final.LoosingTeam.values[0]`), the **Qualifier 2** loser (third place), and the **Eliminator** loser (fourth). Those four cells are then overwritten with `df3.at[wining_team, 'SeasonPosition'] = 'Winner'`, and so on — `at` being the label-based scalar accessor for a single cell.

Two technical warnings surface in the output. The `append` used to build `LoosingTeam` is deprecated (use `pd.concat`), and the task note explains that editing a *view* raises a `SettingWithCopyWarning`, which is why the code begins with `df = matches[...].copy()`. For season 2022 the table correctly marks Gujarat Titans as `Winner` and Rajasthan Royals as `Runner`, with the rest carrying their points rank. This final question stitches together everything: filtering, derived columns, ranking, targeted `.at` assignment, and careful copy semantics.

In [ ]:
# code here
def point_table_extension(season):
    df3 = point_table(season)
    df = matches[matches.Season == season].copy()
    df3["SeasonPosition"] = df3.Points.rank(ascending=False, method= 'first').astype('object')
    df["LoosingTeam"] = df[df.WinningTeam == df.Team1]["Team2"].append(df[df.WinningTeam == df.Team2]["Team1"])
    final = df[df["MatchNumber"] == "Final"]
    wining_team = final.WinningTeam.values[0]
    runner = final.LoosingTeam.values[0]
    df3.at[wining_team, "SeasonPosition"]="Winner"
    df3.at[runner, "SeasonPosition"]="Runner"
    q = df[df["MatchNumber"] == "Qualifier 2"]
    e = df[df["MatchNumber"] == "Eliminator"]
    third = q.LoosingTeam.values[0]
    fourth = e.LoosingTeam.values[0]
    df3.at[third, "SeasonPosition"] = "Third"
    df3.at[fourth, "SeasonPosition"] = "Fourth"
    return df3
point_table_extension("2022")
    

C:\Users\rajti\AppData\Local\Temp\ipykernel_10576\575549411.py:6: FutureWarning: The series.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  df["LoosingTeam"] = df[df.WinningTeam == df.Team1]["Team2"].append(df[df.WinningTeam == df.Team2]["Team1"])


                             Matches Played  Matches Won  No Result  Points  \
Team Name                                                                     
Gujarat Titans                           16           12          0      24   
Rajasthan Royals                         17           10          0      20   
Royal Challengers Bangalore              16            9          0      18   
Lucknow Super Giants                     15            9          0      18   
Delhi Capitals                           14            7          0      14   
Punjab Kings                             14            7          0      14   
Sunrisers Hyderabad                      14            6          0      12   
Kolkata Knight Riders                    14            6          0      12   
Chennai Super Kings                      14            4          0       8   
Mumbai Indians                           14            4          0       8   

                            SeasonPosition  
Team N

## Summary

Task 18 exercised the DataFrame across three domains. On the FIFA World Cup data you built a per-team on-target percentage dictionary with a filter-sum-divide loop and sorted it by value; counted and ranked teams with `value_counts().rank(method='first')`; ran the `info`/`describe`/`isna` checklist, filled numeric nulls with means, dropped duplicates and four columns permanently; and added a typed rank column, promoted it to the index with `set_index`, and sorted. On Titanic you contrasted `dropna(subset=['Age'])` with `fillna(20)` — the same gap handled two deliberate ways. On IPL you renamed franchises before analysis, computed per-team win/home/away percentages (excluding no-results, treating `Team1` as home), found the top no-result venue, parsed pseudo-list squad strings to rank final appearances, and built a points table — then extended it with a `SeasonPosition` column using numeric ranks overwritten via `df.at[...]` for Winner, Runner, Third, and Fourth. Cleaning first, then analysing — the recurring law.